In [1]:
!pip install gymnasium tianshou

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.2/319.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.5/668.5 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.3/69.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.1/472.1 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 8.0 MB/s eta 0:00:00
  Attempting uninstall: filelock
    Found existing installation: filelock 4.0.7
    Uninstalling filelock-4.0.7:
      Successfully uninstalled filelock-4.0.7


In [6]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import torch
import torch.nn as nn

class BarrenDesertEnv(gym.Env):
    """
    Custom Barren Desert Navigation Environment.
    S: Start (Safe)
    F: Flat sand (Safe)
    Q: Quicksand (Trap/Death)
    O: Oasis (Goal/Success)
    """
    metadata = {"render_modes": ["ansi"]}

    def __init__(self, size=4, p_slip=0.2):
        super().__init__()
        self.size = size
        self.p_slip = p_slip
        self.action_space = spaces.Discrete(4)
        self.observation_space = spaces.Discrete(size * size)

        self.grid = np.array([
            ['S', 'F', 'F', 'F'],
            ['F', 'Q', 'F', 'Q'],
            ['F', 'F', 'F', 'Q'],
            ['Q', 'F', 'F', 'O']
        ])

        self.start_pos = (0, 0)
        self.agent_pos = self.start_pos

    def _get_obs(self):
        return self.agent_pos[0] * self.size + self.agent_pos[1]

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.agent_pos = self.start_pos
        return self._get_obs(), {}

    def step(self, action):
        if np.random.rand() < self.p_slip:
            action = self.action_space.sample()

        r, c = self.agent_pos
        if action == 0:   # Left
            c = max(0, c - 1)
        elif action == 1: # Down
            r = min(self.size - 1, r + 1)
        elif action == 2: # Right
            c = min(self.size - 1, c + 1)
        elif action == 3: # Up
            r = max(0, r - 1)

        self.agent_pos = (r, c)
        state_type = self.grid[r, c]

        terminated = False
        reward = -0.01

        if state_type == 'O':
            reward = 1.0
            terminated = True
        elif state_type == 'Q':
            reward = -1.0
            terminated = True

        return self._get_obs(), reward, terminated, False, {}

In [17]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from collections import deque

env = BarrenDesertEnv()

class DQN(nn.Module):
    def __init__(self, state_size=16, action_size=4):
        super(DQN, self).__init__()
        self.embedding = nn.Embedding(state_size, 16)
        self.fc1 = nn.Linear(16, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, action_size)

    def forward(self, x):
        if not isinstance(x, torch.Tensor):
            x = torch.tensor(x, dtype=torch.long)
        x = self.embedding(x)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

state_size = 16
action_size = 4
lr = 1e-3
gamma = 0.99
epsilon_start = 1.0
epsilon_end = 0.1
epsilon_decay = 0.995
batch_size = 32
memory_size = 2000
target_update_interval = 10
num_episodes = 500

policy_net = DQN(state_size, action_size)
target_net = DQN(state_size, action_size)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

optimizer = optim.Adam(policy_net.parameters(), lr=lr)
memory = deque(maxlen=memory_size)

def select_action(state, epsilon):
    if random.random() < epsilon:
        return env.action_space.sample()
    else:
        with torch.no_grad():
            state_tensor = torch.tensor([state], dtype=torch.long)
            return policy_net(state_tensor).argmax().item()

epsilon = epsilon_start
success_count = 0

for episode in range(num_episodes):
    state, _ = env.reset()
    total_reward = 0
    terminated = False

    while not terminated:
        action = select_action(state, epsilon)
        next_state, reward, terminated, truncated, _ = env.step(action)

        memory.append((state, action, reward, next_state, terminated))
        state = next_state
        total_reward += reward

        if len(memory) >= batch_size:
            batch = random.sample(memory, batch_size)
            states, actions, rewards, next_states, dones = zip(*batch)

            states_t = torch.tensor(states, dtype=torch.long)
            actions_t = torch.tensor(actions, dtype=torch.long).unsqueeze(1)
            rewards_t = torch.tensor(rewards, dtype=torch.float32)
            next_states_t = torch.tensor(next_states, dtype=torch.long)
            dones_t = torch.tensor(dones, dtype=torch.float32)

            current_q = policy_net(states_t).gather(1, actions_t).squeeze(1)
            with torch.no_grad():
                max_next_q = target_net(next_states_t).max(1)[0]
                expected_q = rewards_t + (gamma * max_next_q * (1 - dones_t))

            loss = nn.MSELoss()(current_q, expected_q)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    epsilon = max(epsilon_end, epsilon * epsilon_decay)

    if episode % target_update_interval == 0:
        target_net.load_state_dict(policy_net.state_dict())

    if total_reward > 0:
        success_count += 1

    if (episode + 1) % 50 == 0:
        print(f"Episode {episode+1}/{num_episodes} | Epsilon: {epsilon:.3f} | Last Episode Reward: {total_reward:.2f}")

print("Training Complete!")

Episode 50/500 | Epsilon: 0.778 | Last Episode Reward: -1.05


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Episode 100/500 | Epsilon: 0.606 | Last Episode Reward: -1.18
Episode 150/500 | Epsilon: 0.471 | Last Episode Reward: -1.23
Episode 200/500 | Epsilon: 0.367 | Last Episode Reward: -1.09
Episode 250/500 | Epsilon: 0.286 | Last Episode Reward: 0.87
Episode 300/500 | Epsilon: 0.222 | Last Episode Reward: 0.61
Episode 350/500 | Epsilon: 0.173 | Last Episode Reward: -1.37
Episode 400/500 | Epsilon: 0.135 | Last Episode Reward: -1.18
Episode 450/500 | Epsilon: 0.105 | Last Episode Reward: -1.43
Episode 500/500 | Epsilon: 0.100 | Last Episode Reward: 0.92
Training Complete!
